In [1]:
!pip install transformers datasets huggingface_hub transformers[torch] accelerate --upgrade

In [2]:
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, Trainer, DataCollatorForLanguageModeling
from datasets import load_dataset
import torch

In [3]:
from huggingface_hub import login

login()

In [4]:
import re
from sklearn.model_selection import train_test_split

In [5]:
f = open("./drive/MyDrive/metriccoders_datasets/history_of_guptaperiod.txt", "r")
text = f.readlines()

In [6]:
print(len(text))

594


In [7]:
def build_text_files(data_text, dest_path):
    f = open(dest_path, 'w')
    data = ''
    for texts in data_text:
        summary = str(texts).strip()
        summary = re.sub(r"\s", " ", summary)
        data += summary + "  "
    f.write(data)

train, test = train_test_split(text,test_size=0.15)


build_text_files(train,'train_dataset.txt')
build_text_files(test,'test_dataset.txt')

print("Train dataset length: "+str(len(train)))
print("Test dataset length: "+ str(len(test)))

Train dataset length: 504
Test dataset length: 90


In [8]:
tokenizer = AutoTokenizer.from_pretrained("gpt2")

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

/usr/local/lib/python3.10/dist-packages/transformers/tokenization_utils_base.py:1601: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be depracted in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [9]:
train_path = "train_dataset.txt"
test_path = "test_dataset.txt"

In [13]:
from transformers import TextDataset, DataCollatorForLanguageModeling
model = AutoModelForCausalLM.from_pretrained("gpt2")

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [19]:
def load_dataset(train_path, test_path, tokeinzer):
  train_dataset = TextDataset(
          tokenizer=tokenizer,
          file_path=train_path,
          block_size=64)
  test_dataset = TextDataset(
          tokenizer=tokenizer,
          file_path=test_path,
          block_size=64)
  data_collator = DataCollatorForLanguageModeling(
          tokenizer=tokenizer, mlm=False,
  )
  return train_dataset, test_dataset, data_collator

train_dataset, test_dataset, data_collator = load_dataset(train_path, test_path, tokenizer)

/usr/local/lib/python3.10/dist-packages/transformers/data/datasets/language_modeling.py:53: FutureWarning: This dataset will be removed from the library soon, preprocessing should be handled with the 🤗 Datasets library. You can have a look at this example script for pointers: https://github.com/huggingface/transformers/blob/main/examples/pytorch/language-modeling/run_mlm.py
  warnings.warn(
Token indices sequence length is longer than the specified maximum sequence length for this model (13538 > 1024). Running this sequence through the model will result in indexing errors


In [20]:
training_args = TrainingArguments(
    output_dir="./gpt2-history-of-guptaperiod",
    overwrite_output_dir=True,
    num_train_epochs=2,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    eval_steps=400,
    save_steps=100,
    save_total_limit=10,
)

trainer = Trainer(
    model=model,
    args=training_args,
    data_collator=data_collator,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
)

In [21]:
trainer.train()

Step,Training Loss


TrainOutput(global_step=14, training_loss=4.291222436087472, metrics={'train_runtime': 11.0419, 'train_samples_per_second': 38.218, 'train_steps_per_second': 1.268, 'total_flos': 13783154688000.0, 'train_loss': 4.291222436087472, 'epoch': 2.0})

In [22]:
trainer.save_model()

In [31]:
input_text = "The Gupta "
input_ids = tokenizer.encode(input_text, return_tensors="pt").to("cuda")

In [32]:
output = model.generate(input_ids, max_length=100, num_return_sequences=1, pad_token_id=tokenizer.eos_token_id)

In [33]:
generated_text = tokenizer.decode(output[0], skip_special_tokens=True)
print(generated_text)

The Gupta  (1914-19)                                                                                          
